### Installation
Install the packages required for executing this notebook.

In [20]:
# Install the packages
! pip3 install --user --no-cache-dir --upgrade "kfp>2" "google-cloud-pipeline-components>2" \
                                        google-cloud-aiplatform

Check the versions of the packages you installed. The KFP SDK version should be >=1.6.

In [21]:
! python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
! pip3 freeze | grep aiplatform
! python3 -c "import google_cloud_pipeline_components; print('google_cloud_pipeline_components version: {}'.format(google_cloud_pipeline_components.__version__))"

KFP SDK version: 2.17.0
google-cloud-aiplatform==1.165.1
google_cloud_pipeline_components version: 2.22.0


In [22]:
import kfp
import typing
from typing import Dict
from typing import NamedTuple
from kfp import dsl
from kfp.dsl import (Artifact,
                        Dataset,
                        Input,
                        Model,
                        Output,
                        Metrics,
                        ClassificationMetrics,
                        component, 
                        OutputPath, 
                        InputPath)
import google.cloud.aiplatform as aip
from google_cloud_pipeline_components.v1.model import ModelUploadOp
from google_cloud_pipeline_components.v1.endpoint import (EndpointCreateOp,ModelDeployOp)
from google_cloud_pipeline_components.types import artifact_types

#### Project and Pipeline Configurations

In [23]:
#The Google Cloud project that this pipeline runs in.
PROJECT_ID = "your project id"
# The region that this pipeline runs in
REGION = "us-central1"
# Specify a Cloud Storage URI that your pipelines service account can access. The artifacts of your pipeline runs are stored within the pipeline root.
PIPELINE_ROOT = "gs://your temp bucket"   # e.g., gs://temp_de2026

#### Pipeline Component : Train and Test Split

In [24]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn==1.3.2"],
    base_image="python:3.10.7-slim"
)
def train_test_split(dataset: Input[Dataset], dataset_train: Output[Dataset], dataset_test: Output[Dataset]):
    '''train_test_split'''
    import pandas as pd
    import logging 
    import sys
    from sklearn.model_selection import train_test_split as tts
    
    logging.basicConfig(stream=sys.stdout, level=logging.INFO) 
    
    alldata = pd.read_csv(dataset.path, index_col=None)
    train, test = tts(alldata, test_size=0.3)
    train.to_csv(dataset_train.path + ".csv" , index=False, encoding='utf-8-sig')
    test.to_csv(dataset_test.path + ".csv" , index=False, encoding='utf-8-sig')

#### Pipeline Component : Training LogisticRegression

In [25]:
@dsl.component(
    packages_to_install=["pandas", "scikit-learn==1.3.2"],
    base_image="python:3.10.7-slim"
)
def train_lr(features: Input[Dataset], model: Output[Model]):
    '''train a LogisticRegression with default parameters'''
    import pandas as pd
    from sklearn.linear_model import LogisticRegression        
    import pickle 
    
    data = pd.read_csv(features.path+".csv")
    model_lr = LogisticRegression()
    model_lr.fit(data.drop('class',axis=1), data['class'])
    model.metadata["framework"] = "LR"
    file_name = model.path + f".pkl"
    with open(file_name, 'wb') as file:  
        pickle.dump(model_lr, file)   


In [39]:
@dsl.component(
    packages_to_install = [
       "pandas", "scikit-learn==1.3.2", "numpy", "google-cloud-aiplatform"
    ], base_image="python:3.10.7-slim"
)
def lr_model_evaluation(
    project_id: str,
    region: str,
    test_set:  Input[Dataset],
    model_lr: Input[Model],
    thresholds_dict_str: str,
    metrics: Output[ClassificationMetrics],
    kpi: Output[Metrics]
) -> NamedTuple('outputs', approval=bool):
  
    import pandas as pd
    import logging     
    from sklearn.metrics import roc_curve, confusion_matrix, accuracy_score
    import sklearn.metrics as skl_metrics
    import json
    import typing
    from numpy import nan_to_num
    from google.cloud import aiplatform
    import pickle
    from numpy import nan_to_num
    from datetime import datetime
    import random
    idn = random.randint(0,1000)
    
    def threshold_check(val1, val2):
        cond = False
        if val1 >= val2 :
            cond = True
        return cond

    data = pd.read_csv(test_set.path+".csv")
    
     #Loading the saved model with joblib
    m_filename = model_lr.path + ".pkl"
    model = pickle.load(open(m_filename, 'rb'))
    
    y_test = data.drop(columns=["class"])
    y_target = data['class']
    y_pred = model.predict(y_test)    

    y_scores =  model.predict_proba(data.drop(columns=["class"]))[:, 1]
 
    fpr, tpr, thresholds = roc_curve(
         y_true=data['class'], y_score=y_scores, pos_label=True
    )
    
    thresholds = nan_to_num(thresholds)   
    metrics.log_roc_curve(fpr, tpr, thresholds)  
    
    metrics.log_confusion_matrix(
        ['Negative', 'Positive'],
         confusion_matrix(
           data['class'], y_pred
       ).tolist(), 
    )
    
    accuracy = accuracy_score(data['class'], y_pred.round())
    thresholds_dict  = json.loads(thresholds_dict_str)
    model_lr.metadata["accuracy"] = float(accuracy)
    kpi.log_metric("accuracy", float(accuracy))
    outputs = NamedTuple('outputs', approval=bool)
    approval_value = threshold_check(float(accuracy), int(thresholds_dict['roc']))

     # How to start an experiment - just for demonstration 
    aiplatform.init(
       project=project_id,
       location=region,
       experiment="diabetes-lr"
    )
    
    run_id = f"run-{idn}-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    aiplatform.start_run(run_id)
    
    training_metrics = {
        'model_accuracy': skl_metrics.accuracy_score(y_target, y_pred),
        'model_precision': skl_metrics.precision_score(y_target, y_pred, average='macro'),
        'model_recall': skl_metrics.recall_score(y_target, y_pred, average='macro'),
        'model_logloss': skl_metrics.log_loss(y_target, y_pred),
        'model_auc_roc': skl_metrics.roc_auc_score(y_target, y_pred)
    }
    aiplatform.log_metrics(training_metrics)
    
    return outputs(approval_value)

### Upload Model and Metrics to Google Bucket 

In [29]:
@dsl.component(
    packages_to_install=["google-cloud-storage"],
    base_image="python:3.10.7-slim"
)
def upload_model_to_gcs(project_id: str, model_repo: str, model: Input[Model]):
    '''upload model to gsc'''
    from google.cloud import storage   
    import logging 
    import sys
    
    logging.basicConfig(stream=sys.stdout, level=logging.INFO)    
  
    # upload the model to GCS
    client = storage.Client(project=project_id)
    bucket = client.bucket(model_repo)
    blob = bucket.blob('model.pkl')
    source_file_name= model.path + '.pkl'
   
    blob.upload_from_filename(source_file_name)    
    
    print(f"File {source_file_name} uploaded to {model_repo}.")

#### Define the Pipeline

In [40]:
# Define the workflow of the pipeline.
@kfp.dsl.pipeline(
    name="diabetes-predictor-training-pipeline")
def pipeline(project_id: str, region: str, data_bucket: str, dataset_uri: str, model_repo: str, thresholds_dict_str:str, model_repo_uri:str):    
    
    dataset_op = kfp.dsl.importer(
        artifact_uri=dataset_uri,
        artifact_class=Dataset,
        reimport=False,
    )
     
    train_test_split_op = train_test_split(dataset=dataset_op.output)
        
    training_lr_job_run_op = train_lr(features=train_test_split_op.outputs["dataset_train"])
    
    model_evaluation_op = lr_model_evaluation(
        project_id=project_id,
        region=region,
        test_set=train_test_split_op.outputs["dataset_test"],
        model_lr=training_lr_job_run_op.outputs["model"],
        thresholds_dict_str=thresholds_dict_str, # I deploy the model anly if the model performance is above the threshold
    )
    
    with dsl.If(
        model_evaluation_op.outputs["approval"]== True,
        name="approve-model",
    ):
        upload_model_to_gc_op = upload_model_to_gcs(
            project_id=project_id,
            model_repo=model_repo,
            model=training_lr_job_run_op.outputs['model']
        )    

               
        # Using Google's custom components for uploading and deploying the model. 
        # Important: Do not use these in the assignment. We deploy models into Cloud Run with a CI/CD pipeline
        
        import_unmanaged_model_task = dsl.importer(
            artifact_uri=model_repo_uri,
            artifact_class=artifact_types.UnmanagedContainerModel,
            metadata={
                "containerSpec": {
                    "imageUri": "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",  # see https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers  
                },
            },
        ).after(upload_model_to_gc_op)      
       
        model_upload_op = ModelUploadOp(
            project=project_id,
            display_name="diabetes-prediction-model",
            unmanaged_container_model=import_unmanaged_model_task.outputs["artifact"],
        ).after(import_unmanaged_model_task)       
               
        create_endpoint_op = EndpointCreateOp(
            project=project_id,
            display_name="diabetes-prediction-service",
        ).after(model_upload_op)      
        
        model_deploy_op = ModelDeployOp(
            model=model_upload_op.outputs["model"],
            endpoint=create_endpoint_op.outputs['endpoint'],
            deployed_model_display_name="diabetes-prediction-model",
            dedicated_resources_machine_type="n1-standard-4",
            dedicated_resources_min_replica_count=1,
            dedicated_resources_max_replica_count=1,
            traffic_split={"0": 100},
        ).after(create_endpoint_op)     

#### Compile the pipeline into a JSON file

In [41]:
from kfp import compiler
compiler.Compiler().compile(pipeline_func=pipeline,
        package_path='diabetes_prdictor_training_pipeline.yaml')

#### Submit the pipeline run

In [42]:
import google.cloud.aiplatform as aip

# Before initializing, make sure to set the GOOGLE_APPLICATION_CREDENTIALS
# environment variable to the path of your service account.
aip.init(
    project=PROJECT_ID,
    location=REGION,
)

# Prepare the pipeline job
job = aip.PipelineJob(
    display_name="diabetes-predictor",
    enable_caching=False,
    template_path="diabetes_prdictor_training_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    location=REGION,
    parameter_values={
        'project_id': PROJECT_ID, # makesure to use your project id 
        'region': REGION,  
        'data_bucket': 'de2026_data',  # makesure to use your data bucket name 
        'dataset_uri':'gs://de2026_data/training_set.csv',
        'model_repo':'de2026_models', # makesure to use your model bucket name 
        'thresholds_dict_str':'{"roc":0.8}',
        'model_repo_uri':'gs://de2026_models' # makesure to use your model bucket name 
    }
)

job.run()

Error when trying to get or create a GCS bucket for the pipeline output artifacts
Traceback (most recent call last):
  File "/home/jupyter/.local/lib/python3.12/site-packages/google/cloud/aiplatform/pipeline_jobs.py", line 464, in submit
    gcs_utils.create_gcs_bucket_for_pipeline_artifacts_if_it_does_not_exist(
  File "/home/jupyter/.local/lib/python3.12/site-packages/google/cloud/aiplatform/utils/gcs_utils.py", line 381, in create_gcs_bucket_for_pipeline_artifacts_if_it_does_not_exist
    raise ValueError(
ValueError: Output artifacts bucket "de2026_temp" exists but does not belong to project "de2026-508411". This may indicate a bucket squatting attack. Please provide an explicit output_artifacts_gcs_dir parameter or configure one via aiplatform.init(output_artifacts_gcs_dir='gs://your-bucket').
Creating PipelineJob
PipelineJob created. Resource name: projects/529629719494/locations/us-central1/pipelineJobs/diabetes-predictor-training-pipeline-20261008105430
To use this PipelineJob 

### List all models

In [43]:
DISPLAY_NAME = "diabetes-prediction-model"
! gcloud ai models list --region={REGION} --filter={DISPLAY_NAME}

Using endpoint [https://us-central1-aiplatform.googleapis.com/]
MODEL_ID             DISPLAY_NAME
142822712157405184   diabetes-prediction-model
3621853424301113344  diabetes-prediction-model


### Make an online prediction request

In [44]:
ENDPOINT_NAME="diabetes-prediction-service"
instance = [[1,126,60,0,0,30.1,0.349,47]]  # Prediciton request inputs 
ENDPOINT_ID = !(gcloud ai endpoints list --region=$REGION \
              --format='value(ENDPOINT_ID)'\
              --filter=display_name=$ENDPOINT_NAME \
              --sort-by=creationTimeStamp | tail -1)
ENDPOINT_ID = ENDPOINT_ID[1]

def endpoint_predict(
    project: str, location: str, instances: list, endpoint: str
):
    aip.init(project=project, location=location)

    endpoint = aip.Endpoint(endpoint)

    prediction = endpoint.predict(instances=instances)
    return prediction

endpoint_predict(PROJECT_ID, REGION, instance, ENDPOINT_ID)

Prediction(predictions=[0.0], deployed_model_id='2274876913484824576', metadata=None, model_version_id='1', model_resource_name='projects/529629719494/locations/us-central1/models/3621853424301113344', explanations=None)

### Test the batch prediction

In [ ]:
# Define variables 
job_display_name = "diabetes-prediction-batch-prediction-job"
MODEL_NAME="diabetes-prediction-model"
ENDPOINT_NAME="diabetes-prediction-service"
BUCKET_URI="gs://de2026_data"
input_file_name="test_set_batch_prediction.csv"

# Get model id
MODEL_ID=!(gcloud ai models list --region=$REGION \
           --filter=display_name=$MODEL_NAME)
MODEL_ID=MODEL_ID[2].split(" ")[0]

model_resource_name = f'projects/{PROJECT_ID}/locations/{REGION}/models/{MODEL_ID}'
gcs_source= [f"{BUCKET_URI}/{input_file_name}"]
gcs_destination_prefix=f"{BUCKET_URI}/output"

def batch_prediction_job(
    project: str,
    location: str,
    model_resource_name: str,
    job_display_name: str,
    gcs_source: str,
    gcs_destination_prefix: str,
    machine_type: str,
    starting_replica_count: int = 1, # The number of nodes for this batch prediction job. 
    max_replica_count: int = 1,    
):   
    aip.init(project=project, location=location)

    model = aip.Model(model_resource_name)

    batch_prediction_job = model.batch_predict(
        job_display_name=job_display_name,
        instances_format='csv', #csv
        gcs_source=[f"{BUCKET_URI}/{input_file_name}"],
        gcs_destination_prefix=f"{BUCKET_URI}/output",
        machine_type=machine_type, # must be present      
    )
    batch_prediction_job.wait()
    print(batch_prediction_job.display_name)
    print(batch_prediction_job.state)
    return batch_prediction_job

batch_prediction_job(PROJECT_ID, REGION, model_resource_name, job_display_name, gcs_source, gcs_destination_prefix, machine_type="n1-standard-2")

Creating BatchPredictionJob
BatchPredictionJob created. Resource name: projects/529629719494/locations/us-central1/batchPredictionJobs/4789114080952582144
To use this BatchPredictionJob in another session:
bpj = aiplatform.BatchPredictionJob('projects/529629719494/locations/us-central1/batchPredictionJobs/4789114080952582144')
View Batch Prediction Job:
https://console.cloud.google.com/agent-platform/locations/us-central1/batch-predictions/4789114080952582144?project=529629719494
BatchPredictionJob projects/529629719494/locations/us-central1/batchPredictionJobs/4789114080952582144 current state:
JOB_STATE_RUNNING
BatchPredictionJob projects/529629719494/locations/us-central1/batchPredictionJobs/4789114080952582144 current state:
JOB_STATE_RUNNING
BatchPredictionJob projects/529629719494/locations/us-central1/batchPredictionJobs/4789114080952582144 current state:
JOB_STATE_RUNNING
BatchPredictionJob projects/529629719494/locations/us-central1/batchPredictionJobs/4789114080952582144 curr